In [1]:
import os
import shutil
import random

source_dir = "C:/Users/sv200/Downloads/Project5_Ag_Crop and weed detection/Project5_Ag_Crop and weed detection/Project5_Ag_Crop and weed detection/agri_data/data"
output_dir = "C:/Users/sv200/Downloads/Project5_Ag_Crop and weed detection/yolo_dataset"

# Sanity check the source path actually exists
assert os.path.isdir(source_dir), f"source_dir does not exist: {source_dir}"

print("Creating folders...")
for split in ['train', 'val']:
    os.makedirs(os.path.join(output_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dir, 'labels', split), exist_ok=True)

print("Gathering files...")
# Case-insensitive match for common image extensions
valid_exts = ('.jpg', '.jpeg', '.png')
images = [f for f in os.listdir(source_dir) if f.lower().endswith(valid_exts)]

print(f"Found {len(images)} images in source_dir")
assert len(images) > 0, "No images found — check source_dir path and file extensions"

random.shuffle(images)

split_idx = int(len(images) * 0.8)
train_images = images[:split_idx]
val_images = images[split_idx:]

def copy_files(image_list, split_name):
    copied = 0
    for img in image_list:
        base_name = os.path.splitext(img)[0]
        txt_file = base_name + '.txt'

        img_source = os.path.join(source_dir, img)
        img_dest = os.path.join(output_dir, 'images', split_name, img)
        shutil.copy(img_source, img_dest)
        copied += 1

        txt_source = os.path.join(source_dir, txt_file)
        if os.path.exists(txt_source):
            txt_dest = os.path.join(output_dir, 'labels', split_name, txt_file)
            shutil.copy(txt_source, txt_dest)
    return copied

print("Copying training files (80%)...")
n_train = copy_files(train_images, 'train')
print(f"Copied {n_train} training images")

print("Copying validation files (20%)...")
n_val = copy_files(val_images, 'val')
print(f"Copied {n_val} validation images")

print(" Dataset organized successfully! You are ready for the next step.")

Creating folders...
Gathering files...
Found 1300 images in source_dir
Copying training files (80%)...
Copied 1040 training images
Copying validation files (20%)...
Copied 260 validation images
 Dataset organized successfully! You are ready for the next step.


In [2]:
%%writefile dataset.yaml
path: "C:/Users/sv200/Downloads/Project5_Ag_Crop and weed detection/yolo_dataset"
train: images/train
val: images/val

nc: 2
names: ['crop', 'weed']

Overwriting dataset.yaml


In [3]:
%pip install ultralytics

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: ultralytics in c:\users\sv200\appdata\local\programs\python\python310\lib\site-packages (8.4.60)




[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from ultralytics import YOLO

# Load the model
model = YOLO('yolov8n.pt')

# Train the model (YOLO will read the yaml, and the yaml will send it to OneDrive!)
results = model.train(
    data="dataset.yaml",  
    epochs=50,            
    imgsz=512,            
    batch=16,             
    device=0,             
    plots=True            
)

New https://pypi.org/project/ultralytics/8.4.90 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.60  Python-3.10.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, nam